# Persistent indexes

Retrieval data should be built once and reused for many questions.
This tutorial saves the semantic index, lexical index, and code graph
together, then checks whether they still match the repository.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


## 1. Load an embedding model


In [ ]:
BACKEND = "auto"
ALLOW_DOWNLOAD = True
MODEL_BY_BACKEND = {
    "mlx": "mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ",
    "torch": "Qwen/Qwen3-Embedding-0.6B",
}


In [ ]:
from qachat import choose_backend

selected_backend = choose_backend(BACKEND)
MODEL_ID = MODEL_BY_BACKEND[selected_backend]

print(f"Backend: {selected_backend}")
print(f"Model: {MODEL_ID}")


In [ ]:
from qachat import EmbeddingSession

embedding_session = EmbeddingSession(
    MODEL_ID,
    backend=selected_backend,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")


## 2. Prepare real chunks


In [ ]:
from qachat import scan_repository

report = scan_repository(
    root,
    extensions=[".py"],
    include_paths=["qachat"],
)


In [ ]:
from qachat import (
    extract_source_units,
    read_repository_files,
)

source_files = read_repository_files(report)
units = extract_source_units(source_files)


In [ ]:
from qachat import chunk_source_units

chunks = chunk_source_units(
    units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=300,
    max_tokens=400,
    overlap_tokens=40,
)

print(f"Searchable chunks: {len(chunks)}")


## 3. Build the retrieval structures

QAChat prepares three reusable views of the same repository:

- semantic index: meaning
- lexical index: wording and identifiers
- code graph: deterministic code relationships


In [ ]:
from qachat import build_semantic_index

semantic_index = build_semantic_index(
    chunks,
    embedding_session,
)


In [ ]:
from qachat import build_lexical_index

lexical_index = build_lexical_index(chunks)


In [ ]:
from qachat import build_code_graph

code_graph = build_code_graph(source_files, units)

print(f"Code entities: {len(code_graph.entities)}")
print(f"Code relationships: {len(code_graph.relations)}")


## 4. Save all retrieval data

The semantic index, lexical index, and code graph are stored together.
The local `indexes/` folder is excluded from Git.


In [ ]:
from qachat import prepare_persistent_indexes

indexes = prepare_persistent_indexes(
    semantic_index,
    lexical_index,
    code_graph,
)

print(f"Fingerprint: {indexes.fingerprint}")


In [ ]:
from qachat import save_indexes

INDEX_PATH = root / "indexes" / "qachat.json"
saved_path = save_indexes(indexes, INDEX_PATH)

print(f"Saved: {saved_path}")
print(f"Size: {saved_path.stat().st_size:,} bytes")


The saved file is ordinary JSON. It contains the repository chunks, semantic
vectors, lexical vectors, and code graph together with a fingerprint.

The following preview shows one small example from each part.

In [ ]:
import json

payload = json.loads(INDEX_PATH.read_text(encoding="utf-8"))

preview = {
    "format_version": payload["format_version"],
    "fingerprint": payload["fingerprint"][:16] + "...",
    "chunk": {
        "path": payload["chunks"][0]["path"],
        "kind": payload["chunks"][0]["kind"],
        "symbol": payload["chunks"][0]["symbol"],
    },
    "semantic_embedding": payload["semantic_embeddings"][0][:5],
    "lexical_vector": dict(
        list(payload["lexical_vectors"][0].items())[:5]
    ),
    "code_entity": payload["code_graph"]["entities"][0],
    "code_relation": payload["code_graph"]["relations"][0],
}

print(json.dumps(preview, indent=2))

## 5. Load and validate

Loading restores all three structures without rebuilding embeddings,
TF-IDF weights, or code relationships.


In [ ]:
from qachat import load_indexes

loaded_indexes = load_indexes(INDEX_PATH)

print(
    "Loaded semantic chunks: "
    f"{len(loaded_indexes.semantic.chunks)}"
)
print(
    "Loaded lexical chunks: "
    f"{len(loaded_indexes.lexical.chunks)}"
)
print(
    "Loaded code entities: "
    f"{len(loaded_indexes.code_graph.entities)}"
)
print(
    "Loaded code relationships: "
    f"{len(loaded_indexes.code_graph.relations)}"
)


In [ ]:
from qachat import indexes_are_current

assert loaded_indexes.code_graph == code_graph
assert indexes_are_current(
    loaded_indexes,
    chunks,
    code_graph,
)

print("Saved retrieval data matches the current repository.")


In [ ]:
from dataclasses import replace

changed_chunks = list(chunks)
changed_chunks[0] = replace(
    changed_chunks[0],
    text=changed_chunks[0].text + "\n# changed",
)

assert not indexes_are_current(
    loaded_indexes,
    changed_chunks,
    code_graph,
)

print("A repository change makes the saved data stale.")


## 6. Release the model


In [ ]:
embedding_session.close()

print("Embedding model released.")


## Result

QAChat restored the semantic index, lexical index, and code graph from
one file. A shared fingerprint detects when the repository no longer
matches the saved retrieval data.
